# KG1 V1244 CoT-safe — TREINO vigiado (Claude)
Rota A (aposta de título). **Run all.** Pré-req: Colab A100 + Secret `HF_KEY` (escrita).

**SMOKE primeiro (8 steps)** — valida que treina + sobe adapter. Depois suba `MAX_STEPS` p/ 120-160 e re-rode. O juiz REAL de score é o Notebook B (full947).

In [ ]:
import os, subprocess, sys
print('[1/3] clone repo branch', flush=True)
subprocess.run(['git','clone','--depth','1','--branch','claude/v1244-cot-safe','https://github.com/FELIPEACASTRO/KG1-NVIDIA.git','/content/kg1'], check=True)
os.chdir('/content/kg1')
print('[2/3] deps base', flush=True)
subprocess.run([sys.executable,'-m','pip','install','-q','transformers==4.57.6','peft==0.19.1','accelerate==1.13.0','bitsandbytes','safetensors','huggingface_hub','hf_xet'], check=False)
print('[3/3] mamba+causal (source build ~15-25min, normal)', flush=True)
subprocess.run([sys.executable,'-m','pip','install','--no-build-isolation','mamba-ssm==2.3.1'], check=False)
subprocess.run([sys.executable,'-m','pip','install','--no-build-isolation','causal-conv1d==1.6.1'], check=False)
print('DEPS OK', flush=True)


In [ ]:
from google.colab import userdata
import os
for k in ['HF_KEY','HF_TOKEN','HUGGINGFACE_TOKEN']:
    try:
        v=userdata.get(k)
        if v: os.environ['HF_TOKEN']=v; os.environ['HF_KEY']=v; break
    except Exception: pass
assert os.environ.get('HF_TOKEN'), 'Defina HF_KEY no Colab Secrets (com escrita)'
print('HF token OK', flush=True)


In [ ]:
import os
os.environ['DATA_FILE']='/content/kg1/artifacts/v1244_cot_safe/v1244_micro_consolidation_train.jsonl'
os.environ['VAL_FILE']='/content/kg1/artifacts/v1244_cot_safe/v1244_scorelive_evalset_170.jsonl'
os.environ['INIT_ADAPTER_REPO']='felipesp1983/kg1-recovered-v291-v290-checkpoint6-submit086'
os.environ['INIT_ADAPTER_REVISION']='f4134a6d223249d27be2f1c5d94ed59d118d1ce5'
os.environ['REQUIRE_INIT_ADAPTER']='1'
os.environ['LORA_R']='32'; os.environ['LORA_ALPHA']='32'
os.environ['MAX_STEPS']='8'   # SMOKE primeiro (8 steps). Apos validar, suba p/ 120-160 e re-rode.
os.environ['NUM_EPOCHS']='1'; os.environ['EVAL_EVERY_STEPS']='4'; os.environ['SAVE_EVERY_STEPS']='4'
os.environ['LEARNING_RATE']='3e-6'; os.environ['FINAL_LEARNING_RATE']='1e-6'
os.environ['OUTPUT_REPO']='felipesp1983/kg1-v1244-cot-candidate'
os.environ['UPLOAD_TO_HF']='1'
os.environ['REQUIRE_OFFSET_MASK']='1'
print('V1244 env set (SMOKE 8 steps). Dataset=micro 979. ATENCAO: este e o teste da verdade.', flush=True)


In [ ]:
import subprocess, sys, os, time
os.chdir('/content/kg1')
# --- LIVE-LOG: stream stdout -> HF kg1-live-logs a cada 60s (Claude monitora AO VIVO via API) ---
os.environ['KG1_LIVE_LOG_HF_REPO']='felipesp1983/kg1-live-logs'
os.environ['KG1_LIVE_LOG_HF_REPO_TYPE']='dataset'
os.environ['RUN_ID']='v1244_train_'+time.strftime('%Y%m%d_%H%M%S')
# require=1 GARANTE o streaming (valida token+repo+1o upload e aborta se streaming quebrar).
# TREINO REAL LONGO (2h+, MAX_STEPS 120-160): troque p/ '0' — assim um hiccup de upload NAO aborta
# o treino (o wrapper tem retry 60s e os logs locais sempre sao salvos).
os.environ.setdefault('KG1_REQUIRE_LIVE_LOG_UPLOAD','1')
print('LIVE RUN_ID=', os.environ['RUN_ID'], '-> HF:', os.environ['KG1_LIVE_LOG_HF_REPO']+'/colab/'+os.environ['RUN_ID'], flush=True)
# trainer (env-driven) rodado VIA wrapper realtime: streaming + status.json + watchdog
r=subprocess.run([sys.executable,'scripts/kg1_colab_realtime_runner.py','--','python','scripts/hf_job_train_v90.py'])
print('RETURN_CODE=', r.returncode, flush=True)
print('Se RC=0 e adapter subiu p/ OUTPUT_REPO -> rode o NOTEBOOK B p/ medir ACC real no full947.', flush=True)
